<a href="https://colab.research.google.com/github/KHADIJA2008-KB/flyrank-in-A1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KHADIJA2008-KB/redesigned-guacamole/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

base = "hf://datasets/FlyRank/internship-warehouse"
print("Connected.")

Connected.


In [ ]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [ ]:
cols = con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(cols["column_name"].tolist())


['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page (content_hash_id), on one day (report_date), within `month=2026-03` —
using `fact_content_daily_performance`, a mid-panel month rather than the sealed `_sample`
(June 2026, reserved as a future outcome window).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context (join keys):** report_date, client_hash_id, content_hash_id, month

**Features (observed, knowable same-day):** gsc_impressions, gsc_clicks, gsc_avg_position,
ga4_sessions, ga4_engaged_sessions

**Label/proxy:** a simple placeholder decline signal for this exercise (see Section 3's trap)
— a real future-looking label comes in a later week, once I have multiple months to compare.

**Excluded, with why:** the ai_* columns (ai_chatgpt, ai_perplexity, etc.) — excluded from this
lane's core features because they're extremely sparse (per the lane guide, ~30k rows out of
79M have any AI session data) and belong to a different freestyle lane, not Refresh/Content
Opportunity Scoring.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

This section does the heavy lifting — three separate small query cells:

1. Grain check — group by content_hash_id + report_date, confirm count per group = 1
2. Row count + date span — count rows and min/max report_date for your March slice
3. Availability check — filter with ga4_data_available IS TRUE (or whichever boolean flag applies), show rows before/after

Then a 5-feature frame, each with one "available at decision time because…" line, then the deliberate leak: add a label-derived column (like trend_pct, mirroring notebook 2's leakage demo), watch a quick score jump, then remove it.

1. gsc_impressions — knowable same-day: an observed search-console metric for that date
2. gsc_clicks — knowable same-day: an observed search-console metric for that date
3. gsc_avg_position — knowable same-day: computed from that day's search results only
4. ga4_sessions — knowable same-day: an observed analytics metric for that date
5. ga4_engaged_sessions — knowable same-day: an observed analytics metric for that date

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


q1 = f"""
SELECT content_hash_id, report_date, COUNT(*) AS n
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 10
"""
con.sql(q1)  # empty result confirms grain: one row per page per day

q2 = f"""
SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(q2)

q3 = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_gsc
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(q3)

features_df = con.sql(f"""
SELECT
  content_hash_id, client_hash_id, report_date,
  gsc_impressions, gsc_clicks, gsc_avg_position,
  ga4_sessions, ga4_engaged_sessions
FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
LIMIT 5000
""").df()

features_df.head(10)

,content_hash_id,client_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,2026-03-01,20,0,3.350000,<NA>,<NA>
1,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,1,0,0.000000,<NA>,<NA>
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,125,1,4.928000,<NA>,<NA>
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,2026-03-01,7,0,4.000000,<NA>,<NA>
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,11,0,2.272727,<NA>,<NA>
5,content_36c36abc7650d7af,client_73cda7b4e4f265ea,2026-03-01,239,1,7.347280,<NA>,<NA>
6,content_a7da352b73b02668,client_73cda7b4e4f265ea,2026-03-01,191,0,7.832461,<NA>,<NA>
7,content_05434271b257bb68,client_73cda7b4e4f265ea,2026-03-01,55,0,3.272727,<NA>,<NA>
8,content_d056587ff7faca0c,client_73cda7b4e4f265ea,2026-03-01,77,0,5.636364,<NA>,<NA>
9,content_bfd1e41c2af250c8,client_73cda7b4e4f265ea,2026-03-01,2,0,4.500000,<NA>,<NA>


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# Placeholder label for this exercise: is this page currently low-visibility?
features_df["fake_label"] = (features_df["gsc_avg_position"] > 20).astype(int)

# The leak: a feature derived straight from the label
features_df["leaky_feature"] = features_df["gsc_avg_position"]

X_leak = features_df[["gsc_impressions", "gsc_clicks", "ga4_sessions",
                       "ga4_engaged_sessions", "leaky_feature"]].fillna(0)
y = features_df["fake_label"]

clf_leak = DecisionTreeClassifier(max_depth=2).fit(X_leak, y)
print("Score WITH leak:", clf_leak.score(X_leak, y))

# Remove it, honest score
X_clean = features_df[["gsc_impressions", "gsc_clicks", "ga4_sessions",
                        "ga4_engaged_sessions"]].fillna(0)
clf_clean = DecisionTreeClassifier(max_depth=2).fit(X_clean, y)
print("Score WITHOUT leak:", clf_clean.score(X_clean, y))

Score WITH leak: 1.0
Score WITHOUT leak: 0.8544


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice is an unbalanced panel — client_has_gsc/client_has_ga4 show that not every client
has both data sources, and gsc_data_available/ga4_data_available vary row by row depending on
when each client's tracking started. A single month also can't reveal trend or seasonality —
only a snapshot of March 2026.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ✔] Every section above is filled — markdown thinking AND the code that backs it
- [ ✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✔] No client names, URLs, or private queries anywhere
- [ ✔] My claims use careful words: observed, measured, directional, decision-support
- [ ✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.